# 🔥 Notebook 6: Advanced Cache Patterns

Production caching faces challenges that basic patterns don't address. Let's learn how to handle cache stampedes, hot keys, and versioning.

## Learning Objectives

By the end of this notebook, you'll understand:
- Cache stampede and how to prevent it
- Hot key problem and solutions
- Request coalescing
- Cache versioning for safe invalidation

---

🔍 **Open RedisInsight** at http://localhost:5540 to watch cache operations!

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [ ]:
import redis
import json
import time
import random
import threading
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, Callable

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Redis connected")
except:
    print("❌ Redis not running. Start with: docker compose up -d")

## ⚡ Cache Stampede Problem

In [ ]:
print("⚡ Cache Stampede Problem")
print("=" * 60)
print("""
WHAT HAPPENS:
─────────────────────────────────────────────────────────────

1. Popular cache key expires
   └─► All 10,000 concurrent requests see cache MISS

2. All 10,000 requests hit database simultaneously
   └─► Database gets crushed! 💥

3. Database slows down or crashes
   └─► Entire site goes down 😱

Timeline:
─────────────────────────────────────────────────────────────
  Time 0: Cache key exists, everyone happy ✅
  Time T: TTL expires, key deleted
  Time T+1ms: 10,000 requests all see MISS
  Time T+2ms: 10,000 DB queries start
  Time T+100ms: Database CPU at 100% 🔥
""")

In [ ]:
db_queries = 0
lock = threading.Lock()

def expensive_db_query(key: str) -> str:
    global db_queries
    with lock:
        db_queries += 1
    time.sleep(0.05)
    return f"data_for_{key}"

def naive_cache_get(key: str) -> str:
    cached = r.get(key)
    if cached:
        return cached
    
    value = expensive_db_query(key)
    r.setex(key, 60, value)
    return value

print("🔬 Simulating Cache Stampede")
print("=" * 60)

r.delete("stampede:test")
db_queries = 0

print("\n50 concurrent requests hitting expired cache key...")

with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(naive_cache_get, "stampede:test") for _ in range(50)]
    results = [f.result() for f in futures]

print(f"\n📊 Results:")
print(f"   DB queries executed: {db_queries}")
print(f"   Expected (with proper handling): 1")
print(f"\n💥 {db_queries - 1} unnecessary DB queries!")

# If the naive version ever stops stampeding, the next three sections are
# solving a problem the notebook never showed. Fail here instead.
assert db_queries > 10, (
    f"expected a herd of near-50 redundant DB queries, only {db_queries} got "
    f"through — the stampede is no longer being reproduced"
)

## 🔒 Solution 1: Distributed Lock

In [ ]:
import uuid

# Releasing a lock you no longer own is THE classic distributed-lock bug:
# if the DB call overruns the lock's 10s TTL, Redis expires it and hands the
# lock to somebody else — and our `finally` then deletes THEIR lock. Guard it
# with a token only this caller knows, and compare-and-delete atomically
# inside Redis (a GET-then-DEL from Python has the same race in miniature).
_release_lock = r.register_script("""
    if redis.call('get', KEYS[1]) == ARGV[1] then
        return redis.call('del', KEYS[1])
    end
    return 0
""")


def cache_get_with_lock(key: str) -> str:
    cached = r.get(key)
    if cached:
        return cached

    lock_key = f"lock:{key}"
    token = uuid.uuid4().hex                       # proof that WE hold this lock
    acquired = r.set(lock_key, token, nx=True, ex=10)

    if acquired:
        try:
            value = expensive_db_query(key)
            r.setex(key, 60, value)
            return value
        finally:
            _release_lock(keys=[lock_key], args=[token])   # release only if still ours
    else:
        for _ in range(100):
            time.sleep(0.01)
            cached = r.get(key)
            if cached:
                return cached
        # Waited a full second and nothing appeared: the lock holder probably
        # died. Fall through to our own query rather than hanging forever — an
        # escape hatch is mandatory, or one crashed process stalls everyone.
        return expensive_db_query(key)

print("🔒 Solution: Distributed Lock")
print("=" * 60)

r.delete("stampede:locked_test")
r.delete("lock:stampede:locked_test")
db_queries = 0

print("\n50 concurrent requests with lock protection...")

with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(cache_get_with_lock, "stampede:locked_test") for _ in range(50)]
    results = [f.result() for f in futures]

print(f"\n📊 Results:")
print(f"   DB queries executed: {db_queries}")

assert db_queries == 1, (
    f"the lock is supposed to admit exactly ONE request to the database, "
    f"{db_queries} got through"
)
assert all(v == "data_for_stampede:locked_test" for v in results), (
    "every waiter must end up with the same value the owner computed"
)
print(f"   ✅ Only ONE query hit the database, and all 50 callers got the value.")

## 🎲 Solution 2: Probabilistic Early Refresh

In [ ]:
print("🎲 Probabilistic Early Refresh")
print("=" * 60)
print("""
Instead of every request discovering the expiry at the same instant,
each request rolls a die and MAY rebuild the entry before it expires.

The rule below (one common formulation):
─────────────────────────────────────────────────────────────
    elapsed = base_ttl - remaining_ttl
    P(refresh) = (elapsed / base_ttl) * BETA        with BETA = 0.1

So for a 60-second TTL:
    50s remaining (10s old)  →  (10/60) * 0.1 =  1.7%
    30s remaining (30s old)  →  (30/60) * 0.1 =  5.0%
    10s remaining (50s old)  →  (50/60) * 0.1 =  8.3%
     0s remaining (expired)  →  100% — but by now somebody almost
                                certainly refreshed it already

The refreshes get spread over the TTL window instead of piling up
on one millisecond. That is the whole trick.
""")

BETA = 0.1   # how eagerly we refresh; higher = smoother but more DB load

def cache_get_with_early_refresh(key: str, base_ttl: int = 60) -> str:
    cached = r.get(key)
    ttl = r.ttl(key)

    if cached and ttl > 0:
        elapsed = base_ttl - ttl
        refresh_probability = max(0, elapsed / base_ttl) * BETA

        if random.random() < refresh_probability:
            value = expensive_db_query(key)
            r.setex(key, base_ttl, value)
            return value
        return cached

    value = expensive_db_query(key)
    r.setex(key, base_ttl, value)
    return value

In [ ]:
# Smoke test against the real Redis: does the function behave at all?
# (A sequential loop can never produce a stampede — one request at a time is
# the definition of no contention — so this only proves the code runs. The
# next cell is where we actually measure the effect.)
r.delete("early:test")
db_queries = 0
r.setex("early:test", 5, "initial_value")

for _ in range(400):
    cache_get_with_early_refresh("early:test", base_ttl=5)
    time.sleep(0.005)   # ~2s of traffic, well inside the 5s TTL window

print(f"📊 Sequential smoke test: {db_queries} DB queries across 400 cache reads")
assert 0 <= db_queries <= 40, (
    f"early refresh should fire occasionally, not constantly; got {db_queries}"
)
print("   ✅ The function runs and refreshes only occasionally.")
print("   ⚠️  But this tells us NOTHING about stampedes — see the next cell.")

### 📏 Does early refresh actually kill the herd? Measure it.

A stampede is a **concurrency** phenomenon: the damage is not how many database
queries you run in total, it is how many run *at the same instant*. So the
number to measure is **peak concurrent DB queries**, and the only way to compare
two strategies fairly is to replay the identical arrival stream through both.

The simulation below does exactly that — seeded, so it prints the same numbers
on your machine as on mine. One hot key, 1,000 requests/sec, a 10-second TTL,
and a database query that takes 50 ms.


In [ ]:
import bisect

def simulate_hot_key(strategy: str, *, duration_ms=60_000, rps=1000.0,
                     ttl_ms=10_000.0, db_ms=50.0, beta=BETA, seed=5) -> dict:
    """Replay one identical arrival stream against a naive vs early-refresh cache.

    The key detail: a refresh only publishes its result after `db_ms`. Every
    request that arrives during that window ALSO misses — which is precisely
    how a herd forms.
    """
    rng = random.Random(seed)
    expires_at = ttl_ms          # start WARM; we're measuring steady state
    running, query_starts = [], []
    t, rate = 0.0, rps / 1000.0

    while True:
        t += rng.expovariate(rate)
        if t >= duration_ms:
            break
        for _start, finish in running:                 # publish finished refreshes
            if finish <= t:
                expires_at = max(expires_at, finish + ttl_ms)
        running = [(s, f) for s, f in running if f > t]

        remaining = expires_at - t
        if remaining > 0:
            if strategy == "early":
                p = max(0.0, (ttl_ms - remaining) / ttl_ms) * beta
                if rng.random() >= p:
                    continue                           # cache hit, done
            else:
                continue                               # naive: always a hit
        running.append((t, t + db_ms))                 # off to the database
        query_starts.append(t)

    # Peak concurrency = the size of the herd.
    query_starts.sort()
    peak = max(
        (bisect.bisect_right(query_starts, s + db_ms) - bisect.bisect_left(query_starts, s)
         for s in query_starts),
        default=0,
    )
    return {"total": len(query_starts), "peak": peak}


print("📏 60s of 1,000 rps against ONE hot key (10s TTL, 50ms DB query)")
print("=" * 68)

naive = simulate_hot_key("naive")
early = simulate_hot_key("early")

print(f"{'strategy':<26}{'total DB queries':>18}{'peak concurrent':>18}")
print("-" * 68)
print(f"{'naive TTL':<26}{naive['total']:>18}{naive['peak']:>18}")
print(f"{'probabilistic refresh':<26}{early['total']:>18}{early['peak']:>18}")

print(f"\n💥 Every time the TTL rolls over, the naive cache lands {naive['peak']} queries on")
print(f"   the database at once — {naive['total']} redundant queries a minute, all of them")
print(f"   computing the exact same value.")
print(f"🎲 Early refresh peaks at {early['peak']}: a {naive['peak'] / early['peak']:.0f}x smaller herd.")

assert naive["peak"] >= 20, (
    f"the naive cache must visibly stampede for this comparison to mean "
    f"anything; peak was only {naive['peak']}"
)
assert early["peak"] * 5 < naive["peak"], (
    f"early refresh should flatten the herd; got peak {early['peak']} vs "
    f"naive {naive['peak']}"
)

print("\n📐 BETA is the knob — and its cost depends on how hot the key really is:")
print(f"   {'':<12}{'1,000 rps':>22}{'50 rps':>22}")
print(f"   {'':<12}{'total':>10}{'peak':>12}{'total':>10}{'peak':>12}")
for beta in (0.02, 0.05, 0.1, 0.2):
    hot = simulate_hot_key("early", beta=beta, rps=1000.0)
    warm = simulate_hot_key("early", beta=beta, rps=50.0)
    print(f"   BETA={beta:<7}{hot['total']:>10}{hot['peak']:>12}"
          f"{warm['total']:>10}{warm['peak']:>12}")

naive_warm = simulate_hot_key("naive", rps=50.0)
early_warm = simulate_hot_key("early", beta=0.2, rps=50.0)
print(f"\n   (naive at 50 rps: total={naive_warm['total']}, peak={naive_warm['peak']})")
print(f"   👉 At 50 rps with BETA=0.2 early refresh does MORE total work than the")
print(f"      naive cache ({early_warm['total']} vs {naive_warm['total']} queries) and still flattens the peak.")
print(f"      You are buying a smooth load curve with wasted refreshes. That's the deal.")
assert early_warm["total"] > naive_warm["total"], (
    "at light traffic, eager refreshing is supposed to cost extra queries — "
    "if it doesn't, this trade-off paragraph is wrong"
)

print()
print("⚠️  Honest caveat: this linear ramp is RATE-BLIND. It does not know how")
print("    many requests per second are hitting the key, so the same BETA")
print("    refreshes a hot key far more often than a lukewarm one. The")
print("    principled version (XFetch, Vattani et al.) instead stores how long")
print("    the recompute TOOK and refreshes when")
print("        now - delta * beta * ln(random()) >= expiry")
print("    which is self-tuning: expensive keys start refreshing earlier.")

## 🤝 Request Coalescing (single-flight)

**Problem**: even WITH a distributed lock, many app *processes* each do their
own lock dance. And *within* one process, 100 threads asking for the same
expired key will still each try to acquire the lock one by one.

**Fix**: inside each process, collapse concurrent calls for the same key into
*one* in-flight fetch. Every thread waits on the same `Future`. This is called
**single-flight** or **request coalescing**.

```
Without coalescing:    thread1 → DB
                       thread2 → DB   (same query!)
                       thread3 → DB

With coalescing:       thread1 → DB
                       thread2 ┐
                       thread3 ┴── await thread1's result
```


In [ ]:
# In-process request coalescing using a dict of Futures.
from concurrent.futures import Future, TimeoutError as FutureTimeout

class SingleFlightCache:
    def __init__(self, redis_client):
        self.redis = redis_client
        self.in_flight: dict[str, Future] = {}
        self.lock = threading.Lock()

    def get(self, key, loader):
        cached = self.redis.get(key)
        if cached:
            return cached

        # Is someone already fetching this key? Attach to their future.
        with self.lock:
            fut = self.in_flight.get(key)
            if fut is None:
                fut = Future()
                self.in_flight[key] = fut
                owner = True
            else:
                owner = False

        if owner:
            try:
                value = loader(key)           # the ONE real DB call
                self.redis.setex(key, 60, value)
            except BaseException as exc:
                # MANDATORY. A Future that is never completed blocks every
                # waiter forever — a single failing query would hang the whole
                # process. Failures have to be shared as carefully as results.
                fut.set_exception(exc)
                raise
            else:
                fut.set_result(value)
                return value
            finally:
                with self.lock:
                    self.in_flight.pop(key, None)
        else:
            return fut.result()               # wait for the owner

# Demo: 100 threads all ask for the same cold key.
r.delete("coalesce:test")
db_queries = 0
sf_cache = SingleFlightCache(r)

with ThreadPoolExecutor(max_workers=100) as ex:
    futures = [ex.submit(sf_cache.get, "coalesce:test", expensive_db_query)
               for _ in range(100)]
    for f in futures:
        f.result()

print(f"📊 DB queries across 100 concurrent cache misses: {db_queries}")
assert db_queries == 1, (
    f"single-flight is supposed to collapse 100 misses into 1 DB call, "
    f"got {db_queries}"
)
print("   ✅ Exactly 1 — all 100 threads shared one fetch.")


# The interesting half: what happens when that ONE shared fetch fails?
def failing_loader(key):
    time.sleep(0.05)                          # long enough for waiters to attach
    raise RuntimeError("database is down")

r.delete("coalesce:fail")
executor = ThreadPoolExecutor(max_workers=20)
pending = [executor.submit(sf_cache.get, "coalesce:fail", failing_loader)
           for _ in range(20)]

outcomes = []
for f in pending:
    try:
        f.result(timeout=5)
        outcomes.append("returned")
    except RuntimeError:
        outcomes.append("raised")
    except FutureTimeout:
        outcomes.append("HUNG")
executor.shutdown(wait=False)

print(f"\n📊 20 threads sharing a fetch that FAILS: {dict((o, outcomes.count(o)) for o in set(outcomes))}")
assert outcomes.count("HUNG") == 0, (
    "a waiter is stuck: the owner failed without completing the shared Future"
)
assert outcomes.count("raised") == 20, (
    f"every caller should see the failure, got {outcomes.count('raised')}/20"
)
print("   ✅ All 20 saw the error. Nobody is waiting on a Future that will never land.")


## 🔥 Hot Key Problem

In [ ]:
print("🔥 Hot Key Problem")
print("=" * 60)
print("""
SCENARIO: Celebrity posts viral content
─────────────────────────────────────────────────────────────

• Taylor Swift posts on Instagram
• 10 million fans try to view it
• All requests go to ONE cache key: "post:12345"

PROBLEM:
─────────────────────────────────────────────────────────────
• Single Redis node handles ALL requests for that key
• Network bandwidth saturated
• Redis CPU at 100%
• Other keys on same node become slow

SOLUTION: Key Fanout
─────────────────────────────────────────────────────────────
• Store same data under multiple keys
• post:12345:0, post:12345:1, ... post:12345:9
• Clients randomly choose which suffix
• Load distributed across keys/servers
""")

In [ ]:
class HotKeyCache:
    def __init__(self, redis_client, replicas: int = 10):
        self.redis = redis_client
        self.replicas = replicas
        self.key_access_counts = {}

    def _is_hot_key(self, key: str) -> bool:
        count = self.key_access_counts.get(key, 0)
        return count > 100

    def _get_replica_key(self, key: str) -> str:
        if self._is_hot_key(key):
            suffix = random.randint(0, self.replicas - 1)
            return f"{key}:{suffix}"
        return key

    def get(self, key: str):
        self.key_access_counts[key] = self.key_access_counts.get(key, 0) + 1
        replica_key = self._get_replica_key(key)
        value = self.redis.get(replica_key)

        if value is None and replica_key != key:
            # The key only JUST crossed the hot threshold, so the fan-out
            # copies don't exist yet — the last `set` happened while it was
            # still cold. Without this fallback every read between "went hot"
            # and "was next written" silently misses, and the cure for the hot
            # key is worse than the disease. Fall back, then back-fill the
            # shard we were sent to.
            value = self.redis.get(key)
            if value is not None:
                self.redis.setex(replica_key, 60, value)
        return value

    def set(self, key: str, value: str, ttl: int = 60):
        if self._is_hot_key(key):
            for i in range(self.replicas):
                self.redis.setex(f"{key}:{i}", ttl, value)
        else:
            self.redis.setex(key, ttl, value)


print("🔬 Hot Key Fanout Demo")
print("=" * 60)

hot_cache = HotKeyCache(r, replicas=5)

# Clean slate so we can see exactly what gets created.
for k in r.keys("viral:post:12345*"):
    r.delete(k)

# 1) First, warm the single key under normal (non-hot) traffic.
hot_cache.set("viral:post:12345", "Taylor Swift's viral post content")
print("\n1) Initial set — traffic is still 'cold', so ONE key exists:")
for key in sorted(r.keys("viral:post:12345*")):
    print(f"   {key}")

# 2) Traffic explodes. After ~100 reads the key is flagged as hot.
values = [hot_cache.get("viral:post:12345") for _ in range(150)]
print(f"\n2) After 150 reads, access count = "
      f"{hot_cache.key_access_counts['viral:post:12345']} (> 100 → hot).")

# The cold→hot transition is where a naive fan-out breaks: reads 101-150 are
# routed to shard keys that nothing has written yet. If the fallback stops
# working, every one of those reads returns None and the "solution" has
# quietly turned a hot key into a cache miss storm.
assert all(v is not None for v in values), (
    f"{values.count(None)} of 150 reads returned nothing during the "
    f"cold→hot transition — the shard fallback is broken"
)
print(f"   ✅ All 150 reads returned data (the {len(r.keys('viral:post:12345:*'))} "
      f"shard keys that exist now were back-filled on the way).")

# 3) Next write fans the value out to N replica keys — load is now sharded.
hot_cache.set("viral:post:12345", "Taylor Swift's viral post content")
print("\n3) After re-setting the now-hot key, it's stored under N suffixes:")
for key in sorted(r.keys("viral:post:12345*")):
    print(f"   {key}")

shard_keys = r.keys("viral:post:12345:*")
assert len(shard_keys) == hot_cache.replicas, (
    f"a hot-key set must write all {hot_cache.replicas} shards, "
    f"found {len(shard_keys)}"
)
print(f"\n✅ Load distributed across {len(shard_keys)} shard keys "
      f"(+ the original, which the fallback still needs).")
print("   👀 Check RedisInsight for the 'viral:post:12345:*' pattern.")
print()
print("⚠️  Real-world caveat: hotness is tracked PER PROCESS here. In prod")
print("    you'd (a) share counts via Redis itself, or (b) always fan out")
print("    keys you know will be hot (celebrity, homepage, trending).")


## 🔢 Cache Versioning

In [ ]:
print("🔢 Cache Versioning")
print("=" * 60)
print("""
PROBLEM: Cache invalidation race conditions
─────────────────────────────────────────────────────────────
1. Request A reads user from DB (name: "Alice")
2. User updates name to "Alicia" in DB
3. Cache invalidated
4. Request A writes stale "Alice" to cache
5. Everyone sees old name! 😱

SOLUTION: Version in cache key
─────────────────────────────────────────────────────────────
• Store version number in DB: users.version
• Cache key includes version: "user:1:v42"
• On update: increment version to v43
• Old cache (v42) becomes unreachable
• No explicit invalidation needed!
""")

In [ ]:
class VersionedCache:
    def __init__(self, redis_client):
        self.redis = redis_client
        self.versions = {}
    
    def _versioned_key(self, key: str) -> str:
        version = self.versions.get(key, 1)
        return f"{key}:v{version}"
    
    def get(self, key: str) -> Optional[str]:
        versioned_key = self._versioned_key(key)
        return self.redis.get(versioned_key)
    
    def set(self, key: str, value: str, ttl: int = 300):
        versioned_key = self._versioned_key(key)
        self.redis.setex(versioned_key, ttl, value)
    
    def increment_version(self, key: str):
        self.versions[key] = self.versions.get(key, 1) + 1
        return self.versions[key]

print("🔬 Cache Versioning Demo")
print("=" * 60)

vcache = VersionedCache(r)

print("\n1. Cache user with version 1:")
vcache.set("user:1", json.dumps({"name": "Alice"}))
print(f"   Key: user:1:v1")
print(f"   Value: {vcache.get('user:1')}")

print("\n2. User updates name, increment version:")
new_version = vcache.increment_version("user:1")
vcache.set("user:1", json.dumps({"name": "Alicia"}))
print(f"   New version: {new_version}")
print(f"   Key: user:1:v{new_version}")
print(f"   Value: {vcache.get('user:1')}")

print("\n3. Old cache key still exists but unreachable:")
old_value = r.get("user:1:v1")
print(f"   user:1:v1 = {old_value}")
print(f"   (Will expire via TTL, no race condition!)")

assert json.loads(vcache.get("user:1"))["name"] == "Alicia", "v2 must serve the new value"
assert json.loads(old_value)["name"] == "Alice", "v1 must still hold the old value"
assert new_version == 2

print("\n💡 No explicit invalidation - old keys just become orphans!")

print("\n" + "=" * 60)
print("⚠️  Two things this toy gets away with and production will not:")
print("=" * 60)
print()
print("1. `self.versions` is a plain Python dict — it is PER PROCESS. Two app")
print("   servers would disagree about the current version and happily read")
print("   each other's orphans. The version has to live somewhere shared: a")
print("   `users.version` column bumped in the same transaction as the write,")
print("   or an atomic Redis counter:")


class SharedVersionedCache(VersionedCache):
    """Same idea, but the version counter is shared by every process."""

    def _versioned_key(self, key: str) -> str:
        version = self.redis.get(f"ver:{key}") or 1
        return f"{key}:v{version}"

    def increment_version(self, key: str):
        return self.redis.incr(f"ver:{key}")      # atomic, cluster-wide


svc = SharedVersionedCache(r)
for stale_key in r.keys("user:2:v*"):   # clean slate, so re-runs behave
    r.delete(stale_key)
r.set("ver:user:2", 1)
svc.set("user:2", json.dumps({"name": "Bob"}))
assert r.exists("user:2:v1"), "the shared counter should have produced a v1 key"

bumped = svc.increment_version("user:2")
print(f"\n   after one bump the shared counter says v{bumped}")
assert bumped == 2
assert svc.get("user:2") is None, (
    "bumping the version must make the OLD entry unreachable, not overwrite it"
)

svc.set("user:2", json.dumps({"name": "Bobby"}))
assert json.loads(svc.get("user:2"))["name"] == "Bobby"
assert json.loads(r.get("user:2:v1"))["name"] == "Bob", "v1 is orphaned, not deleted"
print("   ✅ old key orphaned, new key readable, and every process agrees on v2.")

print()
print("2. A cache FILL must use the version captured when the READ STARTED.")
print("   If you re-read the version just before writing back, you will stamp")
print("   data you fetched at v1 with the label v2 — which resurrects exactly")
print("   the race (Notebook 5, Scenario C) that versioning was meant to kill:")
print()
print("       version = get_version(key)        # ← capture ONCE, up front")
print("       row     = db.read(key)")
print("       cache.set(f'{key}:v{version}', row)   # ← reuse it, never re-read")
print()
print("   Orphans are the cost of this pattern: they sit in Redis until their")
print("   TTL runs out. Always set one, or a hot key with frequent writes will")
print("   quietly fill your cache with unreachable versions.")

## 🧪 Quick Quiz

1. **What causes a cache stampede?**

2. **How does key fanout solve the hot key problem?**

3. **Why does cache versioning avoid race conditions?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Cache stampede cause:")
print("   - Popular key expires")
print("   - All requests see cache miss simultaneously")
print("   - All hit database at once")
print()
print("2. Key fanout for hot keys:")
print("   - Store same data under multiple keys")
print("   - Clients randomly select which key")
print("   - Load distributed across keys/servers")
print()
print("3. Cache versioning avoids races:")
print("   - Version is part of cache key")
print("   - Update increments version")
print("   - Old cache becomes unreachable")
print("   - Late writers can't overwrite new data")

## 🌍 Beyond Redis: CDN & Sharding

Everything in this notebook scales a *single* Redis cluster. Two more tiers
come into play at very large scale — worth knowing because interviews and real
outages love them.

### 🚀 CDN (Content Delivery Network)

A CDN is an HTTP cache that lives *at the edge*, geographically close to users.
Think Cloudflare, Fastly, CloudFront, Akamai. They speak HTTP, so you control
them with headers:

```
Cache-Control: public, max-age=3600, s-maxage=86400, stale-while-revalidate=60
ETag: "v42-abc123"
```

| When to use | Example |
|-------------|---------|
| Static assets (JS/CSS/images) | Every website |
| Public API responses | Product listing JSON |
| User-agnostic HTML | Blog posts, landing pages |

CDNs can absorb traffic that **never reaches your origin at all** — a single
viral tweet with a CDN hits 1 request/sec at your server while serving
millions to end users.

### 🔀 Sharding (partitioning)

When *one* cache node — or *one* database — can't physically hold the data or
absorb the write rate, you split by key:

```
shard = hash(user_id) % N

user_id 42   → shard 2 → redis-2.example.com
user_id 999  → shard 1 → redis-1.example.com
```

- **Redis Cluster** does this automatically (16,384 hash slots across nodes).
- **Postgres** doesn't natively; you use Citus, Vitess-for-Postgres, or
  application-level sharding.
- **Trade-off**: cross-shard joins / transactions get painful — design your
  keys so most queries stay on one shard.

### 🧩 The full picture

```
      ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
      │   Browser    │─────│     CDN      │─────│  Load Bal.   │
      └──────────────┘     └──────────────┘     └──────┬───────┘
                                                       │
                                                 ┌─────┴─────┐
                                                 ▼           ▼
                                         ┌──────────┐  ┌──────────┐
                                         │ App srv  │  │ App srv  │
                                         └────┬─────┘  └────┬─────┘
                                              │             │
                                              ▼             ▼
                                         ┌───────────────────────┐
                                         │   Redis (sharded)     │
                                         └──────────┬────────────┘
                                                    │ on miss
                                                    ▼
                                        ┌─────────────────────────┐
                                        │ Postgres primary        │
                                        │    └── read replicas    │
                                        └─────────────────────────┘
```

Every layer from left to right handles less traffic than the one before it.
That's the whole trick to scaling reads: **keep as many requests as possible
as far from your primary database as possible.**


## 📚 Summary

### Key Takeaways

1. **Cache stampede** - Use locks or early refresh to prevent DB overload
2. **Hot keys** - Fanout across multiple keys for viral content
3. **Request coalescing** - Combine duplicate in-flight requests
4. **Cache versioning** - Avoids invalidation race conditions
5. **Monitor hit rates** - Low hit rate indicates problems

### Interview Tips

> "For hot keys like celebrity posts, I'd use key fanout - storing the same data under multiple keys like `post:123:0` through `post:123:9`. Clients randomly choose which suffix, distributing load across cache nodes."

> "To prevent cache stampedes, I'd use a distributed lock so only one request rebuilds the cache. Other requests wait briefly for the first one to complete. For critical paths, I'd also use probabilistic early refresh."

### The Complete Read Scaling Toolkit

```
┌─────────────────────────────────────────────────────────────┐
│  1. Indexing          → 10x-100x improvement               │
│  2. Denormalization   → Eliminate joins                    │
│  3. Read replicas     → Scale horizontally                 │
│  4. Application cache → Sub-millisecond reads              │
│  5. CDN               → Global edge caching                │
└─────────────────────────────────────────────────────────────┘
```